In [1]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.applications import EfficientNetB0
import os



os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
df_labels = pd.read_csv("/kaggle/input/datasets/tanlikesmath/diabetic-retinopathy-resized/trainLabels.csv")
#print(df_labels.head(7))
df_labels_cleared = df_labels[['image', 'level']]
#print(df_labels_cleared.shape) -- am realizat faptul ca acest index este invizibil pentru model.
#print(df_labels_cleared)
path = "/kaggle/input/datasets/tanlikesmath/diabetic-retinopathy-resized/resized_train/resized_train"
files = glob.glob(os.path.join(path, "**/*.jpeg"),recursive=True)
file_count = len(files)

#print(file_count)  -- avem 35108 imagini pentru antrenare
#print(df_labels_cleared.groupby("level").size())
# 0 - No DR -25802-
# 1 - Mild  -2438-
# 2 - Moderate  -5288-
# 3 - Severe    -872-
# 4 - Proliferative DR  -708-
# plt.figure(figsize=(15,20))
# for i in range(5):
#     referinta = df_labels_cleared[df_labels_cleared["level"] == i].iloc[0]
#     img_nume = str(referinta["image"])
#     if not img_nume.lower().endswith(".jpeg"):
#         img_nume += ".jpeg"
#     img_path = os.path.join(path, img_nume)
#     img = cv2.imread(img_path)
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     #img_blur = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), 10), -4, 128)

#     plt.subplot(1,5,i+1)
#     plt.imshow(img)
#     plt.title(f"Level {i}")
#     plt.axis("off")



# plt.show()
# print(referinta)

file_paths = []

for image_name in df_labels_cleared["image"]:
    name_with_ext = str(image_name) + ".jpeg"
    file_paths.append(os.path.join(path, name_with_ext))



df_labels_cleared["file_paths"] = file_paths

#print(df_labels_cleared.head(6))
#print(df_labels_cleared["file_paths"])
# print(len(x_train),len(y_train), len(x_test), len(y_test))
#print(class_weight_dict)

def accentuare_detalii_zbancioc(img, alfa = 4, sigma=10):
    img_float = img.astype(np.float32)
    Ib = cv2.GaussianBlur(img_float, (0, 0), sigma)
    I_acc = (img_float - Ib) * alfa + 128
    I_acc = np.clip(I_acc, 0, 255).astype(np.uint8)
    return I_acc


def procesare_imagine(file_path):
    img = cv2.imread(file_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (300,300))
    alfa = 4
    sigma = 10
    Ib = cv2.GaussianBlur(img, (0, 0), sigma)
    img = cv2.addWeighted(img, alfa, Ib, -alfa, 128)
    img = img.astype(np.float32) / 255.0
    # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    # img[:,:,0] = clahe.apply(img[:,:,0])
    # img[:,:,1] = clahe.apply(img[:,:,1])
    # img[:,:,2] = clahe.apply(img[:,:,2])
    # img = img / 255.0
    return img



test_img = procesare_imagine("/kaggle/input/datasets/tanlikesmath/diabetic-retinopathy-resized/resized_train/resized_train/81_right.jpeg")
#print(test_img.shape, test_img, test_img.min(), test_img.max())    #224/224 3-RGB iar pixelii sunt de la 0 care este pixel negru pana la 0.964 care este aproape de pixel 255 - alb



df_balansat_700 = pd.DataFrame()



# for level in range(0,5):
#     df_level = df_labels_cleared[df_labels_cleared["level"] == level]
#     numere = min(len(df_level), 3000)
#     df_sample = df_level.sample(n=numere, random_state=41)
#     df_balansat_700 = pd.concat([df_balansat_700, df_sample])
# df_balansat_700 = df_balansat_700.reset_index(drop=True)





for level in [0, 1, 2]:
    df_level = df_labels_cleared[df_labels_cleared["level"] == level]
    df_sample = df_level.sample(n=min(len(df_level), 4000), random_state=41)
    df_balansat_700 = pd.concat([df_balansat_700, df_sample])





for level in [3, 4]:
    df_level = df_labels_cleared[df_labels_cleared["level"] == level]
    df_multiplied = pd.concat([df_level] * 4) 
    df_balansat_700 = pd.concat([df_balansat_700, df_multiplied])
    
df_balansat_700 = df_balansat_700.sample(frac=1).reset_index(drop=True)

#print(df_balansat_700.groupby("level").size()) 
#print(df_balansat_700.head(7))
# 1500 - 0; 1500 - 1; 1500 - 2; 872 - 3; 708 - 4





x_train, x_test, y_train, y_test = train_test_split(df_balansat_700["file_paths"], df_balansat_700["level"], test_size=0.15,stratify=df_balansat_700["level"],random_state=41)

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [320, 320])
    img = tf.cast(img, tf.float32)
    blur = tf.nn.avg_pool2d(tf.expand_dims(img, 0), ksize=11, strides=1, padding='SAME')
    blur = tf.squeeze(blur, 0)
    img = (img - blur) * 4.0 + 128.0
    img = tf.clip_by_value(img, 0.0, 255.0) / 255.0 
    return img, label
    # img = tf.add(tf.multiply(img, 4.0), tf.multiply(blur, -4.0))
    # img = tf.add(img, 128.0)
    # img = tf.clip_by_value(img, 0.0, 255.0) / 255.0
    # return img, label

#print(f"Avem {len(x_train)} imagini de train si {len(x_test)} imagini de test")

clase = np.asarray([0,1,2,3,4])
weight = compute_class_weight(class_weight="balanced", classes=clase, y=y_train)
class_weight_dict = dict(zip(clase,weight))



#print(len(x_train), len(x_test))
# x_train_list = []
# for filepath in x_train:
#     img = procesare_imagine(filepath)
#     x_train_list.append(img)
# x_train_list = np.array(x_train_list)



# x_test_list = []
# for filepath in x_test:
#     img = procesare_imagine(filepath)
#     x_test_list.append(img)
# x_test_list = np.array(x_test_list)



train_ds = tf.data.Dataset.from_tensor_slices((list(x_train), list(y_train)))
train_ds = train_ds.shuffle(len(x_train)).map(load_and_preprocess).batch(16).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((list(x_test), list(y_test)))
test_ds = test_ds.map(load_and_preprocess).batch(16).prefetch(tf.data.AUTOTUNE)



y_train = np.array(y_train)
y_test = np.array(y_test)

#print(test_img.min(), test_img.max())
#print(y_test.shape, y_train.shape)
# am facut si datele ce nu le stie adica labels ca array pentru a putea face comparatia
#print(x_train_list.shape)

base_model = tf.keras.applications.EfficientNetB0(
    weights="imagenet", 
    include_top=False, 
    input_shape=(320, 320, 3)
)



base_model.trainable = True



for layer in base_model.layers[:40]:
    layer.trainable = False


inputs = tf.keras.Input(shape=(320, 320, 3))
x = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomContrast(0.2),
])(inputs)

x = base_model(x, training=True)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation="swish")(x)
x = Dropout(0.5)(x)
output = Dense(5, activation="softmax")(x)



model = Model(inputs = inputs, outputs = output)
model.summary()

# am declarat x variabila de output a modelului, dupa l am facut intr un vector 2D, dupa l am facut sa nu invete 30% 
# (pentru stabilitate sa nu avem overfitting) dupa am conectat toate straturile, dupa am setat 5 outputuri (de la 5 levels 0,1,2,3,4) 
# pentru clasa





model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), 
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"]
)



reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_diabetic_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)



model.fit(train_ds, epochs=35, validation_data=test_ds, class_weight=class_weight_dict, callbacks=[reduce_lr, early_stop])



model.save("retina_test_320_v1.keras")

2026-04-28 09:25:00.114850: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777368300.301564      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777368300.358703      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777368300.795494      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777368300.795531      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777368300.795534      55 computation_placer.cc:177] computation placer alr

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 320, 320, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 320, 320, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 10, 10, 1280)   │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         2,565 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,713,128 (17.98 MB)

...

 Non-trainable params: 58,291 (227.70 KB)

Epoch 1/35


E0000 00:00:1777368406.149773      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1777368413.517275     124 cuda_dnn.cc:529] Loaded cuDNN version 91002


891/891 ━━━━━━━━━━━━━━━━━━━━ 907s 974ms/step - accuracy: 0.2504 - loss: 1.9854 - val_accuracy: 0.2925 - val_loss: 1.8669 - learning_rate: 5.0000e-05
Epoch 2/35
891/891 ━━━━━━━━━━━━━━━━━━━━ 863s 968ms/step - accuracy: 0.3205 - loss: 1.6486 - val_accuracy: 0.3454 - val_loss: 1.4678 - learning_rate: 5.0000e-05
Epoch 3/35
891/891 ━━━━━━━━━━━━━━━━━━━━ 869s 976ms/step - accuracy: 0.3442 - loss: 1.4826 - val_accuracy: 0.3458 - val_loss: 1.4906 - learning_rate: 5.0000e-05
Epoch 4/35
891/891 ━━━━━━━━━━━━━━━━━━━━ 864s 970ms/step - accuracy: 0.3736 - loss: 1.3976 - val_accuracy: 0.3669 - val_loss: 1.4622 - learning_rate: 5.0000e-05
Epoch 5/35
891/891 ━━━━━━━━━━━━━━━━━━━━ 865s 970ms/step - accuracy: 0.3816 - loss: 1.3526 - val_accuracy: 0.3939 - val_loss: 1.3299 - learning_rate: 5.0000e-05
Epoch 6/35
891/891 ━━━━━━━━━━━━━━━━━━━━ 932s 981ms/step - accuracy: 0.3975 - loss: 1.3230 - val_accuracy: 0.3915 - val_loss: 1.2964 - learning_rate: 5.0000e-05
Epoch 7/35
891/891 ━━━━━━━━━━━━━━━━━━━━ 881s 989ms/

KeyboardInterrupt: 